<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/ace_step_1-5-customi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 1. Clone Repository & Patch System
import os
import sys

# 1. Tải code về
if not os.path.exists("Ace-Step-v1.5"):
    print("⬇️ Cloning ACE-Step v1.5 repository...")
    !git clone https://huggingface.co/spaces/ACE-Step/Ace-Step-v1.5
else:
    print("✅ Repository already exists.")

os.chdir("/content/Ace-Step-v1.5")

# 2. Sửa lỗi phiên bản Torch trong requirements.txt
# (File gốc yêu cầu torch>=2.9.1 chưa ra mắt -> Gây lỗi install -> Sửa thành torch thường)
print("🔧 Patching requirements.txt...")
!sed -i 's/torch>=2.9.1/torch/g' requirements.txt

# 3. Bật tính năng Public Link (Share=True)
print("🌍 Enabling Public Share Link...")
!sed -i 's/share=False/share=True/g' app.py

print("✅ BƯỚC 1 HOÀN TẤT.")

⬇️ Cloning ACE-Step v1.5 repository...
Cloning into 'Ace-Step-v1.5'...
remote: Enumerating objects: 1298, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 1298 (delta 0), reused 0 (delta 0), pack-reused 1295 (from 1)
Receiving objects: 100% (1298/1298), 1.57 MiB | 1.70 MiB/s, done.
Resolving deltas: 100% (789/789), done.
🔧 Patching requirements.txt...
🌍 Enabling Public Share Link...
✅ BƯỚC 1 HOÀN TẤT.


In [2]:
# @title 2. Install Dependencies (Lite Version)
import os

if os.getcwd() != "/content/Ace-Step-v1.5":
    os.chdir("/content/Ace-Step-v1.5")

print("📦 Installing Basic Dependencies...")
# Cài các thư viện cần thiết, bỏ qua các thư viện nặng về biên dịch
!pip install -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121

print("🎵 Installing FFmpeg...")
!apt-get install -y ffmpeg

print("✅ BƯỚC 2 HOÀN TẤT. KHÔNG CẦN CÀI NANO-VLLM.")

📦 Installing Basic Dependencies...
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu128
Ignoring torch: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchaudio: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchvision: markers 'sys_platform == "win32"' don't match your environment
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "win32" and python_version == "3.11" and platform_machine == "AMD64"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "linux" and python_version == "3.11"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 72.0 MB/s eta 0:00:00
INFO: 

🎵 Installing FFmpeg...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
✅ BƯỚC 2 HOÀN TẤT. KHÔNG CẦN CÀI NANO-VLLM.


In [ ]:
# @title 3. Launch Interface (FIX AUTH ERROR)
import os
import sys
import torch
import gc
import time

# --- 1. TẠO SWAP RAM (Giữ nguyên để an toàn) ---
if not os.path.exists("/swapfile"):
    print("💾 Creating Swap Memory...")
    !fallocate -l 10G /swapfile
    !chmod 600 /swapfile
    !mkswap /swapfile
    !swapon /swapfile

# --- 2. TẢI MODEL BẰNG DÒNG LỆNH (MẠNH HƠN PYTHON) ---
# Bước này sửa lỗi 401. Ta dùng huggingface-cli để tải thẳng.
print("⬇️ Đang tải Model 0.6B bằng dòng lệnh (Bỏ qua lỗi Token)...")

# Cài đặt thư viện tải nhanh
!pip install -q huggingface_hub

# Tạo thư mục đích
os.makedirs("/content/Ace-Step-v1.5/data/checkpoints", exist_ok=True)

# LỆNH TẢI CHÍNH (Chìa khóa nằm ở đây)
# Nó sẽ tải thư mục "acestep-5Hz-lm-0.6B" về đúng chỗ mà App cần
!huggingface-cli download ACE-Step/Ace-Step-1.5 \
    --include "acestep-5Hz-lm-0.6B/*" \
    --local-dir /content/Ace-Step-v1.5/data/checkpoints \
    --local-dir-use-symlinks False

print("✅ Đã tải xong Model 0.6B! (Kiểm tra folder data/checkpoints để chắc chắn)")

# --- 3. CẤU HÌNH MÔI TRƯỜNG ---
if os.path.exists("/content/Ace-Step-v1.5"):
    os.chdir("/content/Ace-Step-v1.5")

# Cấu hình Lite Mode
os.environ["SERVICE_MODE_LM_MODEL"] = "acestep-5Hz-lm-0.6B" # Trỏ đúng tên thư mục vừa tải
os.environ["SERVICE_MODE_BACKEND"] = "pytorch"
os.environ["SERVICE_MODE_DIT_MODEL_2"] = ""
os.environ["MAX_MODEL_LEN"] = "2048"
os.environ["GRADIO_SHARE"] = "True"

# --- 4. DỌN RÁC & CHẠY ---
print("🧹 Cleaning Memory...")
gc.collect()
torch.cuda.empty_cache()

print("="*60)
print("🚀 LAUNCHING ACE-STEP (LITE MODE 0.6B)...")
print("👉 Đã dùng CLI để tải model. Lần này sẽ không báo lỗi Not Found nữa.")
print("🔗 Chờ link Gradio hiện ra...")
print("="*60)

!python app.py

⬇️ Đang tải Model 0.6B bằng dòng lệnh (Bỏ qua lỗi Token)...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/commands/download.py:141: FutureWarning: Ignoring --local-dir-use-symlinks. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Returning existing local_dir `/content/Ace-Step-v1.5/data/checkpoints` as remote repo cannot be accessed in `snapshot_download` (401 Client Error. (Request ID: Root=1-6990a31b-7a47997a44898a2d400e2cda;9f15bf2c-42dd-477b-8d16-1047ab993dcf)

Repository Not Found for url: https://huggingface.co/api/models/ACE-Step/Ace-Step-1.5/revision/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.).
/content/Ace-Step-v1.5/data/checkpo

4444

In [5]:
# @title 4. XÓA FILE LỖI & TẢI LẠI (FIX JSON ERROR)
import os
import shutil
import sys
import gc
import torch
from huggingface_hub import hf_hub_download

print("🚑 ĐANG CẤP CỨU HỆ THỐNG...")

# --- 1. XÓA TOKEN BỊ LỖI (Quan trọng) ---
# Nguyên nhân lỗi 401 là do Colab lưu cache token cũ bị sai.
# Ta xóa biến môi trường này đi để tải với tư cách "Khách vãng lai" (Anonymous)
if "HF_TOKEN" in os.environ:
    del os.environ["HF_TOKEN"]
print("🔓 Đã xóa Token cũ (để tránh lỗi quyền truy cập).")

# --- 2. XÓA FILE CORRUPT (FILE HƯ) ---
target_dir = "/content/Ace-Step-v1.5/data/checkpoints/acestep-5Hz-lm-0.6B"
if os.path.exists(target_dir):
    print(f"🗑️ Đang xóa thư mục lỗi: {target_dir}")
    shutil.rmtree(target_dir) # Xóa sạch không thương tiếc

# --- 3. TẢI LẠI TỪNG FILE (BẰNG THƯ VIỆN CHUẨN) ---
print("⬇️ Đang tải lại file sạch (Sẽ mất 1-2 phút)...")
repo_id = "ACE-Step/Ace-Step-1.5"
subfolder = "acestep-5Hz-lm-0.6B"
files_to_download = [
    "config.json",
    "generation_config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json"
]

try:
    for filename in files_to_download:
        print(f"   ⏳ Downloading {filename}...")
        hf_hub_download(
            repo_id=repo_id,
            filename=f"{subfolder}/{filename}",
            local_dir="/content/Ace-Step-v1.5/data/checkpoints",
            local_dir_use_symlinks=False,
            force_download=True
        )
    print("✅ Tải xong! File bảo đảm xịn.")
except Exception as e:
    print(f"❌ Vẫn lỗi tải: {e}")
    # Nếu lỗi này xảy ra, bạn cần kiểm tra lại kết nối internet của Colab

# --- 4. KHỞI ĐỘNG LẠI ---
print("🔄 Restarting App...")
os.system("pkill -9 -f 'python app.py'")
gc.collect()
torch.cuda.empty_cache()

if os.path.exists("/content/Ace-Step-v1.5"):
    os.chdir("/content/Ace-Step-v1.5")

# Cấu hình chuẩn
os.environ["SERVICE_MODE_LM_MODEL"] = "acestep-5Hz-lm-0.6B"
os.environ["SERVICE_MODE_BACKEND"] = "pytorch"
os.environ["SERVICE_MODE_DIT_MODEL_2"] = ""
os.environ["MAX_MODEL_LEN"] = "2048"
os.environ["GRADIO_SHARE"] = "True"

print("="*60)
print("🚀 KHỞI ĐỘNG LẠI (LẦN CUỐI)...")
print("👉 Nếu thấy dòng 'Initializing 5Hz LM' chạy qua mà không đỏ lòm là thành công!")
print("="*60)

!python app.py

🚑 ĐANG CẤP CỨU HỆ THỐNG...
🔓 Đã xóa Token cũ (để tránh lỗi quyền truy cập).
🗑️ Đang xóa thư mục lỗi: /content/Ace-Step-v1.5/data/checkpoints/acestep-5Hz-lm-0.6B
⬇️ Đang tải lại file sạch (Sẽ mất 1-2 phút)...
   ⏳ Downloading config.json...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


❌ Vẫn lỗi tải: Force download failed due to the above error.
🔄 Restarting App...
🚀 KHỞI ĐỘNG LẠI (LẦN CUỐI)...
👉 Nếu thấy dòng 'Initializing 5Hz LM' chạy qua mà không đỏ lòm là thành công!
2026-02-14 16:26:47.934113: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771086407.955054    8627 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771086407.961986    8627 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771086407.979890    8627 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771086407.979913    8627 computation_placer.cc:177] computat